# Lab 2: Build the Retail Sales Database

**Week 2 · Data Engineering Course**

---

## Objectives

By the end of this lab you will have:

1. Designed a 5-table relational schema with foreign keys
2. Created all tables in the correct order (parents before children)
3. Loaded 5 CSV files into PostgreSQL
4. Verified row counts and referential integrity
5. Run your first multi-table JOIN against live data

**Time estimate:** 45–60 minutes

**Tool:** All SQL runs in **pgAdmin → Tools → Query Tool → select `de_course`**

---

## The Schema

```
customers ──< orders ──< order_items >── products
                 │
            salespeople
```

| Table | Rows | Primary key |
|-------|------|-------------|
| customers | 20 | customer_id |
| products | 15 | product_id |
| salespeople | 5 | salesperson_id |
| orders | 50 | order_id |
| order_items | 81 | order_item_id |

> Create tables in this order: customers → salespeople → products → orders → order_items. Tables with foreign keys must be created after the tables they reference.

---

## Setup

1. Open **pgAdmin** from the Start menu
2. Connect to PostgreSQL 16
3. In the left panel, click **`de_course`**
4. Open **Tools → Query Tool**

Confirm you are connected to `de_course` — check the database name shown at the top of the Query Tool window.

---

## Step 1: Create the customers Table

Paste into the Query Tool and press **F5**:

```sql
CREATE TABLE customers (
    customer_id   INTEGER      PRIMARY KEY,
    first_name    VARCHAR(50)  NOT NULL,
    last_name     VARCHAR(50)  NOT NULL,
    email         VARCHAR(100) UNIQUE,
    city          VARCHAR(100),
    signup_date   DATE
);
```

Expected in Messages tab: `CREATE TABLE`

**Why UNIQUE on email?** No two customers should share the same email address. UNIQUE creates an implicit index and rejects duplicate values on INSERT.

---

## Step 2: Create the salespeople Table

```sql
CREATE TABLE salespeople (
    salesperson_id  INTEGER      PRIMARY KEY,
    first_name      VARCHAR(50)  NOT NULL,
    last_name       VARCHAR(50)  NOT NULL,
    region          VARCHAR(50),
    hire_date       DATE
);
```

Expected: `CREATE TABLE`

---

## Step 3: Create the products Table

```sql
CREATE TABLE products (
    product_id    INTEGER          PRIMARY KEY,
    product_name  VARCHAR(100)     NOT NULL,
    category      VARCHAR(50),
    price         NUMERIC(10, 2),
    cost          NUMERIC(10, 2)
);
```

Expected: `CREATE TABLE`

**Why NUMERIC(10, 2)?** NUMERIC stores exact decimal values — important for money. `NUMERIC(10, 2)` means up to 10 total digits with 2 after the decimal point. Avoid FLOAT for currency — floating-point arithmetic can introduce rounding errors.

---

## Step 4: Create the orders Table

```sql
CREATE TABLE orders (
    order_id        INTEGER      PRIMARY KEY,
    customer_id     INTEGER      NOT NULL REFERENCES customers(customer_id),
    salesperson_id  INTEGER      REFERENCES salespeople(salesperson_id),
    order_date      DATE         NOT NULL,
    status          VARCHAR(20)  NOT NULL
);
```

Expected: `CREATE TABLE`

**Note the foreign keys:**
- `customer_id` references `customers` — you cannot insert an order for a non-existent customer
- `salesperson_id` references `salespeople` — it is NOT NULL-able (a salesperson might not always be recorded)

**Test the constraint (optional):**

```sql
-- This should fail:
INSERT INTO orders (order_id, customer_id, salesperson_id, order_date, status)
VALUES (999, 999, 1, '2024-01-01', 'pending');
-- Expected: ERROR: insert or update on table orders violates foreign key constraint

-- Clean up if you ran it:
ROLLBACK;
```

---

## Step 5: Create the order_items Table

```sql
CREATE TABLE order_items (
    order_item_id  INTEGER         PRIMARY KEY,
    order_id       INTEGER         NOT NULL REFERENCES orders(order_id),
    product_id     INTEGER         NOT NULL REFERENCES products(product_id),
    quantity       INTEGER         NOT NULL CHECK (quantity > 0),
    unit_price     NUMERIC(10, 2)  NOT NULL CHECK (unit_price >= 0)
);
```

Expected: `CREATE TABLE`

**CHECK constraints** enforce a rule on a column's value:
- `CHECK (quantity > 0)` prevents inserting an order item with zero or negative quantity
- `CHECK (unit_price >= 0)` prevents negative prices

**Verify all five tables were created:**

```sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
```

Expected: 5 rows — customers, order_items, orders, products, salespeople

---

## Step 6: Load the CSV Files

Load each file using pgAdmin's **Import/Export** feature.

**Load order:** customers → salespeople → products → orders → order_items
(same as CREATE order — parent tables before child tables)

### For each table:

1. In the left panel: expand **de_course → Schemas → public → Tables**
2. Right-click the table name → **Import/Export Data...**
3. Set:
   - Import/Export: **Import**
   - Format: **csv**
   - Encoding: **UTF8**
4. Click the **Options** tab: Header = **Yes**, Delimiter = **,**
5. Browse to the file in `week2/data/`
6. Click **OK**

### Files to load:

| Table | File |
|-------|------|
| customers | `week2/data/customers.csv` |
| salespeople | `week2/data/salespeople.csv` |
| products | `week2/data/products.csv` |
| orders | `week2/data/orders.csv` |
| order_items | `week2/data/order_items.csv` |

### Alternative: COPY commands

Replace `YourName` with your Windows username in each path:

```sql
COPY customers FROM 'C:\Users\YourName\lessons\week2\data\customers.csv' DELIMITER ',' CSV HEADER;
COPY salespeople FROM 'C:\Users\YourName\lessons\week2\data\salespeople.csv' DELIMITER ',' CSV HEADER;
COPY products FROM 'C:\Users\YourName\lessons\week2\data\products.csv' DELIMITER ',' CSV HEADER;
COPY orders FROM 'C:\Users\YourName\lessons\week2\data\orders.csv' DELIMITER ',' CSV HEADER;
COPY order_items FROM 'C:\Users\YourName\lessons\week2\data\order_items.csv' DELIMITER ',' CSV HEADER;
```

Expected output after each: `COPY N` (where N is the row count)

---

## Step 7: Verify the Load

Run each query and check the expected count.

```sql
SELECT 'customers'   AS tbl, COUNT(*) FROM customers
UNION ALL
SELECT 'salespeople',        COUNT(*) FROM salespeople
UNION ALL
SELECT 'products',           COUNT(*) FROM products
UNION ALL
SELECT 'orders',             COUNT(*) FROM orders
UNION ALL
SELECT 'order_items',        COUNT(*) FROM order_items;
```

Expected:

| tbl | count |
|-----|-------|
| customers | 20 |
| salespeople | 5 |
| products | 15 |
| orders | 50 |
| order_items | 81 |

**Verify referential integrity — no orphan order_items:**

```sql
-- Should return 0 rows
SELECT oi.order_item_id
FROM order_items oi
LEFT JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_id IS NULL;
```

---

## Step 8: Add Indexes

The primary key columns are already indexed. Add indexes on foreign key columns — these are the most common JOIN and filter targets.

```sql
CREATE INDEX idx_orders_customer_id    ON orders(customer_id);
CREATE INDEX idx_orders_salesperson_id ON orders(salesperson_id);
CREATE INDEX idx_orders_order_date     ON orders(order_date);
CREATE INDEX idx_order_items_order_id  ON order_items(order_id);
CREATE INDEX idx_order_items_product_id ON order_items(product_id);
```

Expected: `CREATE INDEX` five times.

**Verify indexes were created:**

```sql
SELECT indexname, tablename
FROM pg_indexes
WHERE schemaname = 'public'
ORDER BY tablename, indexname;
```

---

## Step 9: First Multi-Table Queries

Run each and observe the output.

**Full order detail — customer, salesperson, products, totals:**

```sql
SELECT
    o.order_id,
    o.order_date,
    c.first_name || ' ' || c.last_name  AS customer,
    s.first_name || ' ' || s.last_name  AS salesperson,
    p.product_name,
    p.category,
    oi.quantity,
    oi.unit_price,
    oi.quantity * oi.unit_price AS line_total
FROM orders o
INNER JOIN customers   c  ON o.customer_id    = c.customer_id
INNER JOIN salespeople s  ON o.salesperson_id = s.salesperson_id
INNER JOIN order_items oi ON o.order_id       = oi.order_id
INNER JOIN products    p  ON oi.product_id    = p.product_id
ORDER BY o.order_date, o.order_id
LIMIT 20;
```

**Revenue by category:**

```sql
SELECT
    p.category,
    COUNT(DISTINCT o.order_id)               AS orders_containing,
    SUM(oi.quantity)                          AS units_sold,
    SUM(oi.quantity * oi.unit_price)          AS total_revenue
FROM order_items oi
INNER JOIN products p ON oi.product_id = p.product_id
INNER JOIN orders   o ON oi.order_id   = o.order_id
GROUP BY p.category
ORDER BY total_revenue DESC;
```

**Top 5 customers by spend:**

```sql
SELECT
    c.first_name || ' ' || c.last_name AS customer,
    c.city,
    COUNT(DISTINCT o.order_id)         AS order_count,
    SUM(oi.quantity * oi.unit_price)   AS total_spent
FROM customers c
INNER JOIN orders o      ON c.customer_id  = o.customer_id
INNER JOIN order_items oi ON o.order_id    = oi.order_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.city
ORDER BY total_spent DESC
LIMIT 5;
```

---

## Troubleshooting

**`ERROR: relation does not exist`**
You are not connected to `de_course`. Check the database dropdown at the top of the Query Tool.

**`ERROR: insert or update on table orders violates foreign key constraint`** when loading orders.csv
A `customer_id` or `salesperson_id` in orders.csv does not exist in the parent table. Make sure you loaded customers.csv and salespeople.csv before orders.csv.

**`COPY 0` — no rows loaded**
The file path is wrong. Try moving the CSV file to your Desktop and updating the path. On Windows, both `\\` and `/` work in paths.

**`ERROR: duplicate key value violates unique constraint`**
The table already has data. Clear it first:

```sql
TRUNCATE TABLE order_items, orders, products, salespeople, customers RESTART IDENTITY CASCADE;
```

Then reload all five CSVs.

**Table visible in pgAdmin but not in the left panel**
Right-click **Tables** → **Refresh**.